In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from rich import print as rich_print
# from IPython.display import HTML, display

from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
    build_type1_cage_lindblad_construction,
)
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.open_system import (
    LindbladEvolutionOptions,
    analyze_lindblad_evolution,
    initial_density_matrix,
    random_pure_state,
    pure_density_matrix,
    run_quantum_jump_trajectory,
)
from qlinks.visualizer import (
    HamiltonianGraphStyle,
    LiouvillianGraphVisualizer,
    StochasticSchrodingerGraphVisualizer,
)

## Model Definition

In [ ]:
model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=-1.0,
    coup_pot=0.7,
)

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

## Search for caged states

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = 0
record = search_result[signature, record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected caged state

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
    ),
)

rich_print(report.to_rich())

## Lindblad dynamics

In [ ]:
problem = build_type1_cage_lindblad_construction(
    model=model,
    build_result=build_result,
    cage_state=record.full_state,
    classification_report=report,
    z_value=signature[1],  # infer automatically
    monitor_source="reduced_iz_operators",
    reduced_iz_monitor_content="offdiagonal_only",
    reduced_iz_monitor_decomposition="exact_support",
    jump_operator_design="hamiltonian_outside_monitor_inside",
    jump_plaquette_policy="outside_or_crossing",
    recycling_jump_source="local_rdm_two_pattern",
    max_recycling_jumps_per_region=1,
    check_liouvillian=True,
)

rich_print(problem.to_rich())

In [ ]:
diagnostics = problem.diagnose_dark_subspace(
    hamiltonian=build_result.hamiltonian,
    check_liouvillian_spectrum=False,
    # max_liouvillian_dense_dimension=4096,
)

rich_print(diagnostics.to_rich())

In [ ]:
diagnostics = problem.diagnose_absorbing_projector_symmetry(
    hamiltonian=build_result.hamiltonian,
)

rich_print(diagnostics.to_rich())

### 1. Exact method

In [ ]:
gamma = 1.0
eta = 0.0
times = np.linspace(0.0, 100.0, 201)

density_matrix_initial = eta * pure_density_matrix(record.full_state) + (1 - eta) * initial_density_matrix(
    build_result.hamiltonian.shape[0],
    kind="mixed",
    rng=0,
)

evolution = problem.evolve(
    hamiltonian=build_result.hamiltonian,
    density_matrix_initial=density_matrix_initial,
    times=times,
    options=LindbladEvolutionOptions(
        method="auto",
        backend="scipy",
        rk4_step_policy="adaptive",
    ),
)

diagnostics = analyze_lindblad_evolution(
    evolution.density_matrices,
    target_state=problem.cage_state,
    hamiltonian=build_result.hamiltonian,
    jumps=problem.jumps,
)

print(diagnostics.fidelities[-1])
print(diagnostics.purities[-1])
print(diagnostics.lindblad_residuals[-1])

plt.plot(diagnostics.fidelities, linestyle="--", marker=".")
plt.plot(diagnostics.purities, linestyle="--", marker=".")
plt.grid()

### 2. Stochastic Schrodinger method

In [ ]:
times = np.linspace(0.0, 20.0, 201)
psi0 = random_pure_state(basis.n_states, rng=0)

trajectory = run_quantum_jump_trajectory(
    hamiltonian=hamiltonian_matrix,
    jumps=list(problem.jumps),
    state_initial=psi0,
    times=times,
    rng=0,
    max_jump_probability=0.005,
    store_states=True,
    normalize_each_step=True,
    adaptive_time_step=True,
)

## Visualization

### 1. Stochastic Schrodinger animation of a single trajector

In [ ]:
import matplotlib


matplotlib.use("TkAgg")  # Force interactive GUI window

states = np.asarray(trajectory.states, dtype=np.complex128)

visualizer = StochasticSchrodingerGraphVisualizer.from_trajectory(
    times=trajectory.times,
    states=states,
    hamiltonian=hamiltonian_matrix,
    jump_operators=list(problem.jumps),
    basis_labels=[f"|{index}>" for index in range(basis.n_states)],
    style=HamiltonianGraphStyle(
        figure_size=(12.0, 16.0),
        label_vertices=True,
        vertex_size=18.0,
        colorbar=False,
        edge_colorbar=False,
    ),
)

anim = visualizer.animate(
    layout="kk",
    color_by="amplitude_real",
    edge_color_by="weight_complex",
    interval=300,
    repeat=True,
)

plt.show()
# display(HTML(anim.to_jshtml()))

In [ ]:
from matplotlib.animation import FFMpegWriter

# Save as MP4 using FFMpegWriter
writer = FFMpegWriter(fps=24, metadata=dict(artist='Me'), bitrate=1800)
anim.save("/Users/tandaolin/Downloads/animation.mp4", writer=writer)

### 2. Liouvillian graph

[!WARNING] This is expensive !!

In [ ]:
lio_vis = LiouvillianGraphVisualizer.from_liouvillian(
    problem.build_liouvillian(
        hamiltonian_matrix,
        sparse_format="csr"
    ),
    hilbert_dim=basis.n_states,
    density_matrix=pure_density_matrix(record.full_state),
    style=HamiltonianGraphStyle(
        label_vertices=True,
        vertex_size=30.0,
        edge_width=2.5,
        colorbar=True,
        edge_colorbar=False,
        figure_size=(6.0, 5.0),
    ),
)

lio_vis.plot(
    backend="networkx",
    layout="circle",
    color_by="state_amplitude_real",
    edge_color_by="weight_complex",
    title="Liouvillian graph: |rho_ij| nodes, complex L edges",
)